<a href="https://colab.research.google.com/github/wested-kerih/GitHub-Repository/blob/main/Phase_3_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib

url = "https://raw.githubusercontent.com/wested-kerih/GitHub-Repository/refs/heads/main/online_gaming_behavior_insights.csv"
df = pd.read_csv(url)

le = LabelEncoder()
df['EngagementLevel_encoded'] = le.fit_transform(df['EngagementLevel'])

df_encoded = pd.get_dummies(df, columns=['Gender', 'Location', 'GameGenre', 'GameDifficulty'])

X = df_encoded.drop(['PlayerID', 'EngagementLevel', 'EngagementLevel_encoded'], axis=1)
y = df_encoded['EngagementLevel_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5,
                                   min_samples_leaf=2, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance', fontsize=16)

ax1 = axes[0, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Low', 'Medium', 'High'],
            yticklabels=['Low', 'Medium', 'High'])
ax1.set_title(f'Confusion Matrix - Accuracy: {accuracy*100:.1f}%')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')

ax2 = axes[0, 1]
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-10:]
top_features = []
feature_names = X.columns.tolist()
for i in indices:
    name = feature_names[i]
    if len(name) > 30:
        name = name[:27] + "..."
    top_features.append(name)
ax2.barh(range(len(indices)), importances[indices])
ax2.set_yticks(range(len(indices)))
ax2.set_yticklabels(top_features)
ax2.set_xlabel('Importance')
ax2.set_title('Top 10 Features')
ax2.invert_yaxis()

ax3 = axes[1, 0]
classes = ['Low', 'Medium', 'High']
train_counts = np.bincount(y_train)
test_counts = np.bincount(y_test)
x = np.arange(len(classes))
width = 0.35
ax3.bar(x - width/2, train_counts, width, label='Training', alpha=0.8)
ax3.bar(x + width/2, test_counts, width, label='Testing', alpha=0.8)
ax3.set_xlabel('Engagement Level')
ax3.set_ylabel('Number of Players')
ax3.set_title('Class Distribution')
ax3.set_xticks(x)
ax3.set_xticklabels(classes)
ax3.legend()

ax4 = axes[1, 1]
class_accuracies = []
for i in range(3):
    mask = y_test == i
    if sum(mask) > 0:
        class_acc = accuracy_score(y_test[mask], y_pred[mask])
        class_accuracies.append(class_acc)
ax4.bar(classes, class_accuracies, color=['red', 'orange', 'green'], alpha=0.7)
ax4.set_ylim(0, 1)
ax4.set_ylabel('Accuracy')
ax4.set_title('Accuracy by Class')
for i, v in enumerate(class_accuracies):
    ax4.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=300, bbox_inches='tight')

joblib.dump(rf_model, 'gaming_engagement_model.pkl')
joblib.dump(le, 'label_encoder.pkl')

feature_dict = {
    'features': X.columns.tolist(),
    'feature_importance': importances.tolist()
}
import json
with open('feature_info.json', 'w') as f:
    json.dump(feature_dict, f)

metrics = {
    'accuracy': float(accuracy),
    'confusion_matrix': cm.tolist(),
    'class_report': classification_report(y_test, y_pred,
                                        target_names=['Low', 'Medium', 'High'],
                                        output_dict=True)
}
with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f)

report_content = f"""
GAMING ENGAGEMENT PREDICTION MODEL
Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)
Training Samples: {X_train.shape[0]}
Testing Samples: {X_test.shape[0]}
Number of Features: {X.shape[1]}

Top Features:
"""
top_5_idx = np.argsort(importances)[-5:][::-1]
for i, idx in enumerate(top_5_idx, 1):
    feature_name = feature_names[idx]
    importance = importances[idx]
    report_content += f"{i}. {feature_name}: {importance:.4f}\n"

report_content += f"""
Confusion Matrix:
Low      {cm[0,0]} {cm[0,1]} {cm[0,2]}
Medium   {cm[1,0]} {cm[1,1]} {cm[1,2]}
High     {cm[2,0]} {cm[2,1]} {cm[2,2]}

Files Created:
gaming_engagement_model.pkl
model_performance.png
feature_info.json
model_metrics.json
"""

with open('model_report.txt', 'w') as f:
    f.write(report_content)

print(f"Accuracy: {accuracy*100:.1f}%")
print(f"Best Feature: {feature_names[top_5_idx[0]]}")
print("Model saved as gaming_engagement_model.pkl")

plt.show()

NameError: name 'LabelEncoder' is not defined